In [ ]:
import igraph
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import pynauty
from dataclasses import astuple

from qlinks.symmetry.gauss_law import GaussLaw
from qlinks.lattice.component import Site
from qlinks.lattice.square_lattice import SquareLattice, Plaquette
from qlinks.symmetry.automorphism import Automorphism
from qlinks.open_sys.lindbladian import *

from utils import setup_igraph, setup_model

np.set_printoptions(threshold=np.inf)
pd.set_option("display.max_rows", None)

In [ ]:
def ham0(coup_rk):
    _lattice = SquareLattice(4, 4)
    _gauss_law = GaussLaw.from_staggered_charge_distri(4, 4, flux_sector=(0, 0))
    _basis = _gauss_law.solve()

    _kinetic_term = sp.csr_array((_basis.n_states, _basis.n_states), dtype=float)
    _potential_term = sp.csr_array((_basis.n_states, _basis.n_states), dtype=float)
    _hamiltonian = sp.csr_array((_basis.n_states, _basis.n_states), dtype=float)


    # for i, corner_site in enumerate([(0, 1), (0, 3), (2, 0), (2, 2)]):
    #     plaquette = Plaquette(_lattice, Site(*corner_site))
    #     flipper, flip_counter = plaquette[_basis], (plaquette**2)[_basis]
    #     _kinetic_term += flipper
    #     _potential_term += flip_counter
    #     _hamiltonian += (
    #         -1 * flipper + coup_rk * (flip_counter - 4 * sp.identity(_basis.n_states))  # type: ignore[index]
    #     )

    plaquette_ops = {}
    plaquette_sq_ops = {}

    for corner_site in [(0, 1), (0, 3), (2, 0), (2, 2)]:
        plaquette = Plaquette(_lattice, Site(*corner_site))
        plaquette_ops[corner_site] = plaquette[_basis]           # P_(x,y)
        plaquette_sq_ops[corner_site] = (plaquette**2)[_basis]   # P_(x,y)^2

    # potential term stays the same: sum of P^2
    _potential_term = sum(plaquette_sq_ops.values())

    # kinetic term becomes:
    # P^2_(0,1) @ P_(0,3) + P_(0,1) @ P^2_(0,3)
    # + P^2_(2,0) @ P_(2,2) + P_(2,0) @ P^2_(2,2)
    _kinetic_term = (
        plaquette_sq_ops[(0, 1)] @ plaquette_ops[(0, 3)]
        + plaquette_ops[(0, 1)] @ plaquette_sq_ops[(0, 3)]
        + plaquette_sq_ops[(2, 0)] @ plaquette_ops[(2, 2)]
        + plaquette_ops[(2, 0)] @ plaquette_sq_ops[(2, 2)]
    )

    _hamiltonian = (
        -1 * _kinetic_term
        + coup_rk * (_potential_term - 4 * sp.identity(_basis.n_states))
    )

    return _basis, _hamiltonian, _kinetic_term, _potential_term

In [ ]:
coup_rk = -0.7
basis, hamiltonian, kinetic_term, potential_term = ham0(coup_rk)

evals, evecs = np.linalg.eigh(hamiltonian.toarray())

plt.matshow(evecs)
plt.colorbar()

In [ ]:
plt.plot(evals, linestyle="--", marker="o")
plt.grid()

In [ ]:
aut = Automorphism(-kinetic_term)
aut.joint_partition.keys()

g = nx.from_scipy_sparse_array(-kinetic_term)
ig = igraph.Graph.from_networkx(g)

ntg = pynauty.Graph(
    ig.vcount(),
    directed=False,
    adjacency_dict=nx.to_dict_of_lists(g),
)
aut_gp = pynauty.autgrp(ntg)[0]

# perm_gp = PermutationGroup([Permutation(p) for p in aut_gp])

In [ ]:
g = nx.from_scipy_sparse_array(-kinetic_term)

# highlight = list(aut.degree_partition.values())
# highlight = perm_gp.orbits()
# highlight = list(nx.bipartite.sets(g))
highlight = []

# highlight_color = list(mcolors.TABLEAU_COLORS.values())
# highlight_color = list(mcolors.CSS4_COLORS.values())
# cmap = plt.get_cmap('Set3')
# highlight_color = [mcolors.to_hex(cmap(i)) for i in range(cmap.N)]
# cmap = plt.get_cmap('Set2')
# highlight_color += [mcolors.to_hex(cmap(i)) for i in range(cmap.N)]
# highlight_color *= 2000
highlight_color = [
#     "dimgray",
#     "whitesmoke",
    # "deepskyblue",
    # "yellowgreen",
    # "aqua",
    # "pink",
    # "tomato",
    # "royalblue",
    # "blueviolet",
    # "cornflowerblue",
    # "limegreen",
]

ig = setup_igraph(g, highlight, highlight_color)

# degree = np.array(list(dict(g.degree).values()))
# (bipartite, types) = ig.is_bipartite(return_types=True)
# nodes = [int(n) for n in list(sub_sub_ig.vs["label"])]
# outer_boundary = list(nx.node_boundary(g, nodes))
# sub_ig = ig.induced_subgraph(np.append(nodes, outer_boundary))

# sub_ig = ig.induced_subgraph(np.where(degree == 12)[0])
# sub_ig = ig.induced_subgraph(aut.type_1_scars((8, 'B'))[0].node_idx)
# fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")
igraph.plot(
    ig,
    # layout=ig.layout_kamada_kawai(),
    # layout=ig.layout_reingold_tilford(root=[0, 25, 50, 75]),
    # layout=ig.layout_bipartite(types=types),
    vertex_size=18,
    vertex_label_size=10,
    # vertex_label_dist=1.5,
    edge_width=0.6,
    # edge_color="darkgray",
)

In [ ]:
sub_components = ig.connected_components(mode="weak")

for i, c in enumerate(sub_components):
    mat = nx.to_numpy_array(ig.subgraph(c).to_networkx())
    if mat.shape[0] > 1:
        print(i, mat.shape[0], mat.shape[0] - np.linalg.matrix_rank(mat))

In [ ]:
sub_ig = ig.subgraph(sub_components[4])
igraph.plot(
    sub_ig,
    layout=sub_ig.layout_kamada_kawai(),
    vertex_size=24,
    vertex_label_size=12,
    edge_width=0.4,
    # edge_color="darkgray",
    # target="qlm_subgraph_4x4_d=8_by_orbits.svg"
)

In [ ]:
evals, evecs = np.linalg.eigh(nx.to_numpy_array(sub_ig.to_networkx()))

In [ ]:
evals

In [ ]:
plt.matshow(evecs)
plt.colorbar()

In [ ]:
# Example: two-level amplitude damping

d = 2

# Hamiltonian
H = sp.csc_matrix(np.array([[0.0, 0.0], [0.0, 1.0]], dtype=np.complex128))

gamma = 0.3

# Jump operator: sqrt(gamma) |0><1|
J = sp.csc_matrix(np.sqrt(gamma) * np.array([[0.0, 1.0], [0.0, 0.0]], dtype=np.complex128))
jumps = [J]

# Initial excited state density matrix
rho0 = np.array([[0.0, 0.0], [0.0, 1.0]], dtype=np.complex128)

times = np.linspace(0.0, 60.0, 201)

# Build Liouvillian
Lio = build_liouvillian(H, jumps, fmt="csc")

# Krylov evolution
rhos_krylov = evolve_liouvillian_krylov(rho0, Lio, times)

# RK4 in Liouville space
rhos_rk4 = evolve_liouvillian_rk4(
    rho0, Lio, times, renormalize_trace=False, enforce_hermiticity=False
)

# Example observable: population of |1>
proj1 = np.array([[0.0, 0.0], [0.0, 1.0]], dtype=np.complex128)

pop1_krylov = [float(np.real(np.trace(rho @ proj1))) for rho in rhos_krylov]
pop1_rk4 = [float(np.real(np.trace(rho @ proj1))) for rho in rhos_rk4]

print("Final trace (Krylov):", trace_of_rho(rhos_krylov[-1]))
print("Final trace (RK4):   ", trace_of_rho(rhos_rk4[-1]))
print("Final pop |1> Krylov:", pop1_krylov[-1])
print("Final pop |1> RK4:   ", pop1_rk4[-1])
print("Final fidelity to ground truth (Krylov):   ", fidelity_pure(np.array([[1], [0]]), rhos_krylov[-1]))
print("Final fidelity to ground truth (RK4):   ", fidelity_pure(np.array([[1], [0]]), rhos_rk4[-1]))

In [ ]:
def prepare_ticqmbs_state(n: int, one_based: bool = False):
    """
    Prepare the sparse state vector

        |psi_tICQMBS> = 1/2 (|5> - |15> - |74> + |77>)

    and its density matrix rho = |psi><psi|.

    Parameters
    ----------
    n : int
        Total Hilbert-space dimension.
    one_based : bool, optional
        If True, interpret |5>, |15>, |74>, |77> as physics-style basis labels
        and convert them to Python's 0-based indices.
        If False, use them directly as 0-based indices.

    Returns
    -------
    psi : scipy.sparse.csr_matrix
        Sparse column vector of shape (n, 1).
    rho : scipy.sparse.csr_matrix
        Sparse density matrix of shape (n, n).
    """
    labels = np.array([5, 15, 74, 77], dtype=int)
    coeffs = np.array([1, -1, -1, 1], dtype=np.complex128) / 2.0

    indices = labels - 1 if one_based else labels

    if np.any(indices < 0) or np.any(indices >= n):
        raise ValueError(f"State labels {labels.tolist()} do not fit in Hilbert space dimension n={n}.")

    # Sparse column vector |psi>
    psi = sp.csr_matrix(
        (coeffs, (indices, np.zeros(len(indices), dtype=int))),
        shape=(n, 1),
        dtype=np.complex128,
    )

    # Sparse density matrix rho = |psi><psi|
    rho = psi @ psi.getH()

    return psi, rho

def random_mixed_density(n, density=0.25, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    def data_rvs(k):
        # real and imaginary parts in [-1, 1]
        return rng.uniform(-1, 1, size=k) + 1j * rng.uniform(-1, 1, size=k)

    A = sp.random(
        n, n,
        density=density,
        data_rvs=data_rvs,
        dtype=np.complex128,
        format="csr"
    )

    rho = A @ A.getH()
    rho = rho / rho.diagonal().sum()
    return rho


coup_j, coup_rk, gamma = (1, -0.7, 1.0)
basis, model = setup_model(
    "qdm", lattice_shape=(4, 4), coup_j=coup_j, coup_rk=coup_rk
)

plaquette_ops = {}
plaquette_sq_ops = {}

for corner_site in model.lattice:
    plaquette = Plaquette(model.lattice, Site(*corner_site))
    plaquette_ops[astuple(corner_site)] = plaquette[basis]           # P_(x,y)
    plaquette_sq_ops[astuple(corner_site)] = (plaquette**2)[basis]   # P_(x,y)^2


jumps = []
jumps.append(
    np.sqrt(gamma) * model.kinetic_term @ (
        -plaquette_ops[(0, 1)]
        + coup_rk * (plaquette_sq_ops[(0, 1)] - sp.identity(basis.n_states))
        - plaquette_ops[(0, 3)]
        + coup_rk * (plaquette_sq_ops[(0, 3)] - sp.identity(basis.n_states))
    )
)
# jumps.append(
#     np.sqrt(gamma) * (plaquette_ops[(0, 3)]) @ model.kinetic_term @ (
#         -plaquette_ops[(0, 1)]
#         + coup_rk * (plaquette_sq_ops[(0, 1)] - sp.identity(basis.n_states))
#         - plaquette_ops[(0, 3)]
#         + coup_rk * (plaquette_sq_ops[(0, 3)] - sp.identity(basis.n_states))
#     )
# )
# jumps.append(
#     np.sqrt(gamma) * (plaquette_ops[(2, 0)]) @ model.kinetic_term @ (
#         -plaquette_ops[(2, 0)]
#         + coup_rk * (plaquette_sq_ops[(2, 0)] - sp.identity(basis.n_states))
#         - plaquette_ops[(2, 2)]
#         + coup_rk * (plaquette_sq_ops[(2, 2)] - sp.identity(basis.n_states))
#     )
# )
jumps.append(
    np.sqrt(gamma) * model.kinetic_term @ (
        -plaquette_ops[(2, 0)]
        + coup_rk * (plaquette_sq_ops[(2, 0)] - sp.identity(basis.n_states))
        - plaquette_ops[(2, 2)]
        + coup_rk * (plaquette_sq_ops[(2, 2)] - sp.identity(basis.n_states))
    )
)

# for site_pair in [[(0, 1), (0, 3), (2, 0), (2, 2)]]:
#     J = sp.csr_array((basis.n_states, basis.n_states), dtype=np.complex128)
#     for i, corner_site in enumerate(site_pair):
#         plaquette = Plaquette(model.lattice, Site(*corner_site))
#         flipper, flip_counter = plaquette[basis], (plaquette**2)[basis]
#         J += -flipper + coup_rk * (flip_counter - sp.identity(basis.n_states))
#     # J += coup_rk * (model.potential_term - 4 * sp.identity(basis.n_states))
#     jumps.append(np.sqrt(gamma) * J)

for i, corner_site in enumerate(
    [(0, 0), (0, 2), (2, 1), (2, 3), (1, 0), (1, 1), (1, 2), (1, 3), (3, 0), (3, 1), (3, 2), (3, 3)]
):
    plaquette = Plaquette(model.lattice, Site(*corner_site))
    flipper, flip_counter = plaquette[basis], (plaquette**2)[basis]
    J = -flipper + coup_rk * flip_counter
    jumps.append(np.sqrt(gamma) * J)

# for i, corner_site in enumerate(
#     [(0, 1), (0, 3), (2, 0), (2, 2)]
# ):
#     plaquette = Plaquette(model.lattice, Site(*corner_site))
#     flipper, flip_counter = plaquette[basis], (plaquette**2)[basis]
#     J = 0.1 * (-flipper + coup_rk * flip_counter)
#     jumps.append(np.sqrt(gamma) * J)



_, rho0 = prepare_ticqmbs_state(basis.n_states)
rng = np.random.default_rng()
eta = 0.0
rho0 = eta * rho0+ (1 - eta) * random_mixed_density(basis.n_states, density=0.25)
times = np.linspace(0.0, 100.0, 201)

# Build Liouvillian
Lio = build_liouvillian(model.hamiltonian, jumps, fmt="csr")

# Krylov evolution
rhos_krylov = evolve_liouvillian_krylov(rho0.toarray(), Lio, times)

# RK4 in Liouville space
# rhos_rk4 = evolve_liouvillian_rk4(
#     rho0.toarray(), Lio, times, renormalize_trace=False, enforce_hermiticity=False
# )
# rhos_rk4 = evolve_matrix_rk4(rho0, model.hamiltonian, jumps, times)


psi0, ground_truth_rho0 = prepare_ticqmbs_state(basis.n_states)
proj1 = sp.identity(basis.n_states) - ground_truth_rho0
pop1_krylov = [float(np.real(np.trace(rho @ proj1))) for rho in rhos_krylov]
# pop1_rk4 = [float(np.real(np.trace(rho @ proj1))) for rho in rhos_rk4]

# they should be nullifier
print("Jump_opt @ |ground_truth_QMBS>:  ", [sp.linalg.norm(jump @ ground_truth_rho0) for jump in jumps])
print("|| Lio @ |final_rho> ||:  ", np.linalg.norm(Lio @ vec(rhos_krylov[-1])))

print("Final trace (Krylov):", trace_of_rho(rhos_krylov[-1]))
# print("Final trace (RK4):   ", trace_of_rho(rhos_rk4[-1]))
print("Final pop on other |v_j> (Krylov):   ", pop1_krylov[-1])
# print("Final pop on other |v_j> (RK4):   ", pop1_rk4[-1])

print("Final fidelity to ground truth (Krylov):   ", fidelity_pure(psi0.reshape((-1, 1)), rhos_krylov[-1]))

In [ ]:
def trace_preservation_residual(L, n):
    I_vec = np.eye(n, dtype=np.complex128).reshape(-1, order="F")
    v = I_vec.conj() @ L
    return np.max(np.abs(v))

trace_preservation_residual(Lio, basis.n_states)

In [ ]:
rho0 = sp.random(
    basis.n_states, basis.n_states, density=0.6, rng=rng, dtype=np.complex128, format="csr"
)
unvec(Lio @ vec(rho0.toarray()), basis.n_states).trace()

In [ ]:
[np.linalg.norm(rho - rho.conj().T) for rho in rhos_krylov]

In [ ]:
# # L_sparse: scipy sparse matrix
# evals = np.linalg.eigvals(Lio.toarray())
#
# plt.figure(figsize=(6, 6))
# plt.scatter(evals.real, evals.imag, s=20)
# plt.axhline(0, color='k', linewidth=0.8)
# plt.axvline(0, color='k', linewidth=0.8)
# plt.xlabel(r'Re$(\lambda)$')
# plt.ylabel(r'Im$(\lambda)$')
# plt.title('Liouvillian spectrum')
# plt.tight_layout()
# plt.show()

In [ ]:
# k = number of eigenvalues you want
k = 50

# which='LR' means largest real part
evals, evecs = spla.eigs(Lio, k=k, which='LR')

plt.figure(figsize=(6, 6))
plt.scatter(evals.real, evals.imag, s=20)
plt.axhline(0, color='k', linewidth=0.8)
plt.axvline(0, color='k', linewidth=0.8)
plt.xlabel(r'Re$(\lambda)$')
plt.ylabel(r'Im$(\lambda)$')
plt.title(f'Partial Liouvillian spectrum (k={k})')
plt.tight_layout()
plt.show()